<a href="https://colab.research.google.com/github/drameyjoshi/tc/blob/main/mqc/CO_molecule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyscf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 MB 29.9 MB/s  0:00:01


In [2]:
import pyscf

# The molecule

The bond length is known to be 112.8 pm, that is $1.128\; Å$. C has a ground state of He$2s^22p^2$. The two p-electrons are unpaired so that the total spin is $S = 1$, making the ground state a triplet. O has a ground state of He$2s^22p^4$. Of the four p electrons, 2 are paired while two are unpaired. As a result, the ground state of O is also a triplet.

In [3]:
# To properly construct and initialize the molecule, we call the build() method.
mol = pyscf.gto.Mole()
mol.atom = "C 0 0 0; O 0 0 1.128"  # Placing them along the z-axis with an experimental bond length (~1.128 Angstroms)
mol.basis = 'sto-3g'
mol.verbose = 3
mol.build()

# Let's print out some properties to verify everything is set up correctly
print(f"Elements in molecule: {mol.elements}")
print(f"Number of electrons: {mol.nelectron}")
print(f"Basis functions count: {mol.nao}")

Elements in molecule: ['C', 'O']
Number of electrons: 14
Basis functions count: 10


# Choice of algorithm


The number of valence electrons in CO molecule are 4 from C and 6 from O, a total of 10. Since this is an even number, we expect all electrons in CO molecule to be paired. Therefore, we can use the restricted Hartree-Fock theory to compute the ground state energy. Let's first use the minimal basis.

The total number of electrons, as printed above, is 14. Therefore we have a $14 \times 14$ determinant.
Since we have 10 basis functions in sto-3g, we can form exactly 10 spatial molecular orbitals. Each spatial molecular orbital $\phi_j$ is a linear combination of all 10 basis functions:

$$\phi_j = \sum_{\mu=1}^{10} C_{\mu j} \chi^{\text{AO}}_\mu \quad (j = 1, 2, \dots, 10)$$.

To construct the spin-orbitals $\chi_i$ used in the Slater determinant, we multiply each of our 10 spatial molecular orbitals $\phi_j$ by one of the two spin functions, $\alpha$ (spin-up) or $\beta$ (spin-down):

$$\chi_{2j-1}(\mathbf{x}) = \phi_j(\mathbf{r}) \alpha(s)$$ $$\chi_{2j}(\mathbf{x}) = \phi_j(\mathbf{r}) \beta(s)$$

Since we have 10 spatial MOs, this gives us exactly 20 possible spin-orbitals:

- 10 spin-up orbitals
- 10 spin-down orbitals

Because Carbon Monoxide has 14 electrons, we choose the 14 lowest-energy spin-orbitals from our pool of 20 available spin-orbitals to populate the ground state Slater determinant.

We completely fill the 7 lowest-energy spatial MOs ($\phi_1$ to $\phi_7$).
This uses exactly 14 spin-orbitals: $\phi_1\alpha, \phi_1\beta, \phi_2\alpha, \phi_2\beta, \dots, \phi_7\alpha, \phi_7\beta$.
So, each of the 14 $\chi_i$ functions in the determinant is indeed made of 10 spatial terms (the linear combination of the 10 atomic basis functions) multiplied by a single spin part ($\alpha$ or $\beta$).

The HF procecure consists in finding $C_{\mu j}$ that minimise $\langle\Psi|H|\Psi\rangle$ where
$$\Psi(\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_{14}) = \frac{1}{\sqrt{14!}} \begin{vmatrix} \chi_1(\mathbf{x}_1) & \chi_2(\mathbf{x}_1) & \dots & \chi_{14}(\mathbf{x}_1) \\ \chi_1(\mathbf{x}_2) & \chi_2(\mathbf{x}_2) & \dots & \chi_{14}(\mathbf{x}_2) \\ \vdots & \vdots & \ddots & \vdots \\ \chi_1(\mathbf{x}_{14}) & \chi_2(\mathbf{x}_{14}) & \dots & \chi_{14}(\mathbf{x}_{14}) \end{vmatrix}$$

In [4]:
# 1. Run Restricted Hartree-Fock (RHF)
print("=== Running RHF ===")
rhf_mol = pyscf.scf.RHF(mol)
rhf_energy = rhf_mol.kernel()

# 2. Run Unrestricted Hartree-Fock (UHF)
print("\n=== Running UHF ===")
uhf_mol = pyscf.scf.UHF(mol)
uhf_energy = uhf_mol.kernel()

# 3. Check Spin Contamination <S^2> for UHF
s2, multip = uhf_mol.spin_square()

print("\n=== Comparison ===")
print(f"RHF Energy: {rhf_energy:.8f} Eh")
print(f"UHF Energy: {uhf_energy:.8f} Eh")
print(f"UHF <S^2> Spin Contamination: {s2:.4f} (0.0000 means it is a pure closed-shell singlet)")

=== Running RHF ===
converged SCF energy = -111.224558695609

=== Running UHF ===
converged SCF energy = -111.224558695609  <S^2> = 8.2627238e-12  2S+1 = 1

=== Comparison ===
RHF Energy: -111.22455870 Eh
UHF Energy: -111.22455870 Eh
UHF <S^2> Spin Contamination: 0.0000 (0.0000 means it is a pure closed-shell singlet)


# Calculations with larger bases

To verify our result, let's test how the energy changes with larger, more accurate basis sets.

In [5]:
basis_sets = ['sto-3g', '3-21g', '6-31g', 'cc-pvdz']

print(f"{'Basis Set':<12} | {'SCF Energy (Eh)':<18} | {'Difference from STO-3G (Eh)':<28}")
print("-" * 65)

sto3g_energy = -111.22455870

for basis in basis_sets:
    test_mol = pyscf.gto.Mole()
    test_mol.atom = "C 0 0 0; O 0 0 1.128"
    test_mol.basis = basis
    test_mol.verbose = 0
    test_mol.build()

    mf = pyscf.scf.RHF(test_mol)
    energy = mf.kernel()
    diff = energy - sto3g_energy
    print(f"{basis:<12} | {energy:<18.8f} | {diff:<28.8f}")

Basis Set    | SCF Energy (Eh)    | Difference from STO-3G (Eh) 
-----------------------------------------------------------------
sto-3g       | -111.22455870      | 0.00000000                  
3-21g        | -112.09329674      | -0.86873804                 
6-31g        | -112.66720456      | -1.44264586                 
cc-pvdz      | -112.74931133      | -1.52475263                 


In [6]:
# Conversion factor from Hartree (a.u.) to eV
HARTREE_TO_EV = 27.211386

sto3g_energy_ev = rhf_energy * HARTREE_TO_EV
ccpvdz_energy_ev = energy * HARTREE_TO_EV

print(f"STO-3G Ground State Energy: {rhf_energy:.8f} a.u. = {sto3g_energy_ev:.4f} eV")
print(f"cc-pVDZ Ground State Energy: {energy:.8f} a.u. = {ccpvdz_energy_ev:.4f} eV")
print(f"Energy difference: {energy - rhf_energy:.8f} a.u. = {(energy - rhf_energy) * HARTREE_TO_EV:.4f} eV")

STO-3G Ground State Energy: -111.22455870 a.u. = -3026.5744 eV
cc-pVDZ Ground State Energy: -112.74931133 a.u. = -3068.0650 eV
Energy difference: -1.52475263 a.u. = -41.4906 eV


# Molecular orbitals

We print the energies of the ten molecular orbitals.

In [7]:
# Print the molecular orbital (MO) energies using the RHF model object
print("=== Molecular Orbital Energies (Hartrees) ===")
for i, energy_val in enumerate(rhf_mol.mo_energy):
    occ = rhf_mol.mo_occ[i]
    print(f"Orbital {i+1:2d}: Energy = {energy_val:11.6f} Hartree | Occupation = {occ}")

=== Molecular Orbital Energies (Hartrees) ===
Orbital  1: Energy =  -20.424353 Hartree | Occupation = 2.0
Orbital  2: Energy =  -11.093412 Hartree | Occupation = 2.0
Orbital  3: Energy =   -1.460134 Hartree | Occupation = 2.0
Orbital  4: Energy =   -0.699485 Hartree | Occupation = 2.0
Orbital  5: Energy =   -0.551233 Hartree | Occupation = 2.0
Orbital  6: Energy =   -0.551233 Hartree | Occupation = 2.0
Orbital  7: Energy =   -0.446474 Hartree | Occupation = 2.0
Orbital  8: Energy =    0.313495 Hartree | Occupation = 0.0
Orbital  9: Energy =    0.313495 Hartree | Occupation = 0.0
Orbital 10: Energy =    1.040435 Hartree | Occupation = 0.0


# Ionisation energy

From the NIST database, the experimentally observed ionisation energy is 14.014 eV. We are getting there but 2 eV is still a significant difference.

In [8]:
ionisation_energy = -rhf_mol.mo_energy[6] * HARTREE_TO_EV
print(f"Ionisation energy in sto-3g basis: {ionisation_energy:.4f} eV")

Ionisation energy in sto-3g basis: 12.1492 eV


## Using larger basis

In [9]:
best_mol = pyscf.gto.Mole()
best_mol.atom = "C 0 0 0; O 0 0 1.128"
best_mol.basis = 'cc-pvdz'
best_mol.verbose = 0
best_mol.build()

print(f"Elements in molecule: {best_mol.elements}")
print(f"Number of electrons: {best_mol.nelectron}")
print(f"Basis functions count: {best_mol.nao}")

best_estimates = pyscf.scf.RHF(best_mol)
best_ground_state = best_estimates.kernel()
# Print the molecular orbital (MO) energies using the RHF model object
print("=== Molecular Orbital Energies (Hartrees) ===")
for i, energy_val in enumerate(best_estimates.mo_energy):
    occ = best_estimates.mo_occ[i]
    print(f"Orbital {i+1:2d}: Energy = {energy_val:11.6f} Hartree | Occupation = {occ}")

Elements in molecule: ['C', 'O']
Number of electrons: 14
Basis functions count: 28
=== Molecular Orbital Energies (Hartrees) ===
Orbital  1: Energy =  -20.669325 Hartree | Occupation = 2.0
Orbital  2: Energy =  -11.366539 Hartree | Occupation = 2.0
Orbital  3: Energy =   -1.523078 Hartree | Occupation = 2.0
Orbital  4: Energy =   -0.802127 Hartree | Occupation = 2.0
Orbital  5: Energy =   -0.634505 Hartree | Occupation = 2.0
Orbital  6: Energy =   -0.634505 Hartree | Occupation = 2.0
Orbital  7: Energy =   -0.548791 Hartree | Occupation = 2.0
Orbital  8: Energy =    0.154722 Hartree | Occupation = 0.0
Orbital  9: Energy =    0.154722 Hartree | Occupation = 0.0
Orbital 10: Energy =    0.382591 Hartree | Occupation = 0.0
Orbital 11: Energy =    0.672223 Hartree | Occupation = 0.0
Orbital 12: Energy =    0.672223 Hartree | Occupation = 0.0
Orbital 13: Energy =    0.726535 Hartree | Occupation = 0.0
Orbital 14: Energy =    1.018459 Hartree | Occupation = 0.0
Orbital 15: Energy =    1.06199

In [10]:
ionisation_energy = -best_estimates.mo_energy[6] * HARTREE_TO_EV
print(f"Ionisation energy in cc-pvdz basis: {ionisation_energy:.4f} eV")

Ionisation energy in cc-pvdz basis: 14.9334 eV
